# 05 Model Interpretability and Business Recommendations

Translate model drivers into investigator-facing prioritization and responsible business recommendations.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Project\Data Science\financial_fraud_detection_&_transaction_intelligence")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import json

import pandas as pd
import plotly.express as px

from src.config import REPORT_DIR, RISK_SCORE_DIR
from src.visualization import APPROVED_PALETTE

In [3]:
permutation = pd.read_csv(REPORT_DIR / "permutation_importance.csv")
native = pd.read_csv(REPORT_DIR / "model_native_feature_importance.csv")
notes = json.loads((REPORT_DIR / "interpretability_notes.json").read_text(encoding="utf-8"))
permutation.head(12), native.head(12), notes

(                             feature  permutation_importance_mean  \
 0   amount_equals_old_origin_balance                 5.196057e-01   
 1          is_origin_account_drained                 1.110223e-16   
 2               origin_balance_ratio                 1.110223e-16   
 3           transfer_or_cashout_flag                 5.551115e-17   
 4               origin_balance_error                 5.551115e-17   
 5                             amount                 0.000000e+00   
 6               origin_balance_delta                 0.000000e+00   
 7                               step                 0.000000e+00   
 8           transaction_type_encoded                 0.000000e+00   
 9                         amount_log                 0.000000e+00   
 10         destination_balance_error                 0.000000e+00   
 11         destination_balance_delta                 0.000000e+00   
 
     permutation_importance_std  
 0                 6.915007e-03  
 1                 1

In [4]:
plot_frame = native.head(12).sort_values("importance")
px.bar(
    plot_frame,
    x="importance",
    y="feature",
    orientation="h",
    title="Model-Native Feature Importance",
    color_discrete_sequence=[APPROVED_PALETTE["accent_teal"]],
)

In [5]:
queue = pd.read_csv(RISK_SCORE_DIR / "investigation_priority_top_1000.csv")
queue[
    [
        "investigation_priority_rank",
        "step",
        "type",
        "amount",
        "predicted_fraud_probability",
        "fraud_risk_score",
        "model_risk_segment",
        "isFlaggedFraud",
        "isFraud",
    ]
].head(20)

,investigation_priority_rank,step,type,amount,predicted_fraud_probability,fraud_risk_score,model_risk_segment,isFlaggedFraud,isFraud
0,3933,742,TRANSFER,1819543.69,1.0,1000.0,Critical,0,1
1,3911,740,TRANSFER,1755647.81,1.0,1000.0,Critical,0,1
2,3929,742,TRANSFER,4009058.39,1.0,1000.0,Critical,0,1
3,3876,736,CASH_OUT,3912252.81,1.0,1000.0,Critical,0,1
4,3932,742,CASH_OUT,652993.91,1.0,1000.0,Critical,0,1
5,3875,736,TRANSFER,3912252.81,1.0,1000.0,Critical,0,1
6,3871,736,TRANSFER,274799.58,1.0,1000.0,Critical,0,1
7,3872,736,CASH_OUT,274799.58,1.0,1000.0,Critical,0,1
8,3858,735,TRANSFER,417103.68,1.0,1000.0,Critical,0,1
9,3859,735,CASH_OUT,417103.68,1.0,1000.0,Critical,0,1


## Recommendations

- Prioritize high-risk TRANSFER and CASH_OUT transactions for review.
- Use model probability and risk score to rank investigation queues by capacity.
- Treat `isFlaggedFraud` as a legacy rule, not the only detection layer.
- Monitor account-drain behavior and amount-equals-origin-balance patterns.
- Frame cost and recovery analysis as scenario assumptions only.
- Recalibrate and validate the model with real investigation outcomes before production use.